In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Dietary"
)

xpt_file = DATA_DIR / "DR2TOT_E.xpt"
csv_file = DATA_DIR / "DR2TOT_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 DR2TOT_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = """
SEQN
WTDRD1
WTDR2D
DR2DRSTZ
DR2EXMER
DRABF
DRDINT
DR2DBIH
DR2DAY
DR2LANG
DR2MNRSP
DR2HELPD
DR2TNUMF
DR2TKCAL
DR2TPROT
DR2TCARB
DR2TSUGR
DR2TFIBE
DR2TTFAT
DR2TSFAT
DR2TMFAT
DR2TPFAT
DR2TCHOL
DR2TATOC
DR2TATOA
DR2TRET
DR2TVARA
DR2TACAR
DR2TBCAR
DR2TCRYP
DR2TLYCO
DR2TLZ
DR2TVB1
DR2TVB2
DR2TNIAC
DR2TVB6
DR2TFOLA
DR2TFA
DR2TFF
DR2TFDFE
DR2TCHL
DR2TVB12
DR2TB12A
DR2TVC
DR2TVD
DR2TVK
DR2TCALC
DR2TPHOS
DR2TMAGN
DR2TIRON
DR2TZINC
DR2TCOPP
DR2TSODI
DR2TPOTA
DR2TSELE
DR2TCAFF
DR2TTHEO
DR2TALCO
DR2TMOIS
DR2TS040
DR2TS060
DR2TS080
DR2TS100
DR2TS120
DR2TS140
DR2TS160
DR2TS180
DR2TM161
DR2TM181
DR2TM201
DR2TM221
DR2TP182
DR2TP183
DR2TP184
DR2TP204
DR2TP205
DR2TP225
DR2TP226
DR2_300
DR2_320Z
DR2_330Z
DR2BWATZ
DR2TWS
""".split()


if df.shape != (9762, 83):
    raise ValueError(
        f"Unexpected DR2TOT_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "DR2TOT_E columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: DR2TOT_E structure = "
    "9,762 rows x 83 columns."
)


# ============================================================
# 4. DEFINE DIETARY MEASUREMENT FIELDS
# ============================================================

#
# ALL of these must remain source-decoded FLOAT values.
#
# NO rounding.
# NO float_format.
# NO Int64 conversion.
# NO recalculation.
#

NUTRIENT_COLUMNS = (
    df.columns[
        df.columns.get_loc("DR2TKCAL"):
        df.columns.get_loc("DR2TP226") + 1
    ]
    .tolist()
)


MEASUREMENT_COLUMNS = (
    [
        "WTDRD1",
        "WTDR2D"
    ]
    +
    NUTRIENT_COLUMNS
    +
    [
        "DR2_320Z",
        "DR2_330Z",
        "DR2BWATZ"
    ]
)


INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in MEASUREMENT_COLUMNS
]


if len(NUTRIENT_COLUMNS) != 65:
    raise ValueError(
        "Unexpected nutrient field count."
    )


if len(MEASUREMENT_COLUMNS) != 70:
    raise ValueError(
        "Unexpected dietary measurement field count."
    )


if len(INTEGER_COLUMNS) != 13:
    raise ValueError(
        "Unexpected integer/code field count."
    )


print(
    "\nNutrient fields preserved AS IS:",
    len(NUTRIENT_COLUMNS)
)

print(
    "Total dietary measurement fields:",
    len(MEASUREMENT_COLUMNS)
)

print(
    "Identifier/code/count fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)


print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print(
    "\nTiny values by variable:"
)


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


if int(
    true_zero_counts.sum()
) != 0:

    raise ValueError(
        "Unexpected true numeric zero(s) "
        "already decoded from XPT."
    )


# ============================================================
# 6. EXPECTED TINY COUNTS
#    ACTUAL ATTACHED DR2TOT_E XPT
# ============================================================

EXPECTED_TINY_COUNTS = {

    "DR2DBIH": 5,
    "DR2TNUMF": 1,

    "DR2TKCAL": 6,
    "DR2TPROT": 7,
    "DR2TCARB": 7,
    "DR2TSUGR": 8,
    "DR2TFIBE": 83,
    "DR2TTFAT": 7,
    "DR2TSFAT": 9,
    "DR2TMFAT": 9,
    "DR2TPFAT": 9,
    "DR2TCHOL": 55,
    "DR2TATOC": 9,

    "DR2TATOA": 6871,

    "DR2TRET": 77,
    "DR2TVARA": 24,
    "DR2TACAR": 989,
    "DR2TBCAR": 119,
    "DR2TCRYP": 835,
    "DR2TLYCO": 2275,
    "DR2TLZ": 110,

    "DR2TVB1": 8,
    "DR2TVB2": 8,
    "DR2TNIAC": 8,
    "DR2TVB6": 8,

    "DR2TFOLA": 7,
    "DR2TFA": 153,
    "DR2TFF": 76,
    "DR2TFDFE": 7,

    "DR2TCHL": 7,
    "DR2TVB12": 25,

    "DR2TB12A": 4893,

    "DR2TVC": 43,
    "DR2TVD": 100,
    "DR2TVK": 12,

    "DR2TCALC": 1,
    "DR2TPHOS": 6,
    "DR2TMAGN": 1,
    "DR2TIRON": 7,
    "DR2TZINC": 3,
    "DR2TCOPP": 1,
    "DR2TSODI": 1,
    "DR2TPOTA": 6,
    "DR2TSELE": 7,

    "DR2TCAFF": 2118,
    "DR2TTHEO": 4152,
    "DR2TALCO": 6869,
    "DR2TMOIS": 1,

    "DR2TS040": 509,
    "DR2TS060": 486,
    "DR2TS080": 269,
    "DR2TS100": 123,
    "DR2TS120": 33,
    "DR2TS140": 11,
    "DR2TS160": 9,
    "DR2TS180": 10,

    "DR2TM161": 30,
    "DR2TM181": 9,
    "DR2TM201": 92,
    "DR2TM221": 2502,

    "DR2TP182": 9,
    "DR2TP183": 9,
    "DR2TP184": 4939,
    "DR2TP204": 301,
    "DR2TP205": 2174,
    "DR2TP225": 2898,
    "DR2TP226": 1744,

    "DR2_320Z": 2004,
    "DR2_330Z": 4159,
    "DR2BWATZ": 5202
}


EXPECTED_TINY_TOTAL = 57565


# ============================================================
# 7. VALIDATE EXACT TINY PATTERN
# ============================================================

actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    missing_columns = (
        set(EXPECTED_TINY_COUNTS)
        -
        set(actual_tiny_counts)
    )


    unexpected_columns = (
        set(actual_tiny_counts)
        -
        set(EXPECTED_TINY_COUNTS)
    )


    count_differences = {
        col: {
            "expected": EXPECTED_TINY_COUNTS.get(col),
            "actual": actual_tiny_counts.get(col)
        }
        for col in (
            set(EXPECTED_TINY_COUNTS)
            &
            set(actual_tiny_counts)
        )
        if (
            EXPECTED_TINY_COUNTS[col]
            !=
            actual_tiny_counts[col]
        )
    }


    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Missing: {sorted(missing_columns)}\n"
        f"Unexpected: {sorted(unexpected_columns)}\n"
        f"Count differences: {count_differences}"
    )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny artifact count."
    )


if len(
    actual_tiny_counts
) != 70:

    raise ValueError(
        "Unexpected number of tiny-value variables."
    )


print(
    "\nPASS: Exactly 57,565 tiny artifacts "
    "across 70 validated variables."
)


# ============================================================
# 8. SAFETY CHECK:
#    TINY VALUES ONLY IN APPROVED ZERO FIELDS
# ============================================================

APPROVED_ZERO_COLUMNS = (
    set(
        NUTRIENT_COLUMNS
    )
    |
    {
        "DR2DBIH",
        "DR2TNUMF",
        "DR2_320Z",
        "DR2_330Z",
        "DR2BWATZ"
    }
)


unexpected_zero_columns = (
    set(
        actual_tiny_counts
    )
    -
    APPROVED_ZERO_COLUMNS
)


if unexpected_zero_columns:

    raise ValueError(
        "STOP: Tiny values found in "
        "unexpected non-approved variables:\n"
        + str(
            sorted(
                unexpected_zero_columns
            )
        )
    )


print(
    "PASS: Tiny artifacts occur only in "
    "validated zero-permitted fields."
)


# ============================================================
# 9. RESTORE ONLY VALIDATED ZEROS
# ============================================================

print(
    "\n--- RESTORING VALIDATED ZEROS ---"
)


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0.0


print(
    "PASS: All 57,565 validated "
    "tiny values restored to zero."
)


# ============================================================
# 10. VERIFY NO TINY VALUE REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "Remaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

print(
    "\n--- SEQN CHECK ---"
)


seqn = df["SEQN"]


if int(
    seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "PASS: SEQN 41475-51623, "
    "no missing, no duplicates."
)


# ============================================================
# 12. DR2DBIH CHECK
# ============================================================

#
# CDC:
#
# Range   = -55 to 162
# Valid   = 7,874
# Missing = 1,888
#
# Attached XPT:
#
# zero = 5
#

s = df["DR2DBIH"]


if int(
    s.notna()
    .sum()
) != 7874:

    raise ValueError(
        "DR2DBIH valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 1888:

    raise ValueError(
        "DR2DBIH missing count differs."
    )


if s.min() != -55:

    raise ValueError(
        "Unexpected DR2DBIH minimum."
    )


if s.max() != 162:

    raise ValueError(
        "Unexpected DR2DBIH maximum."
    )


if int(
    s.eq(0)
    .sum()
) != 5:

    raise ValueError(
        "DR2DBIH restored zero count differs."
    )


print(
    "\nPASS: DR2DBIH"
)

print(
    "      range   = -55 to 162"
)

print(
    "      valid   = 7,874"
)

print(
    "      missing = 1,888"
)

print(
    "      zeros   = 5"
)


# ============================================================
# 13. DR2TNUMF CHECK
# ============================================================

#
# CDC 2007-2008:
#
# Range   = 0 to 56
# Valid   = 7,838
# Missing = 1,924
#
# The one tiny value IS a legitimate zero.
#

s = df["DR2TNUMF"]


if int(
    s.notna()
    .sum()
) != 7838:

    raise ValueError(
        "DR2TNUMF valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 1924:

    raise ValueError(
        "DR2TNUMF missing count differs."
    )


if s.min() != 0:

    raise ValueError(
        "Unexpected DR2TNUMF minimum."
    )


if s.max() != 56:

    raise ValueError(
        "Unexpected DR2TNUMF maximum."
    )


if int(
    s.eq(0)
    .sum()
) != 1:

    raise ValueError(
        "DR2TNUMF zero count differs."
    )


print(
    "PASS: DR2TNUMF"
)

print(
    "      range   = 0 to 56"
)

print(
    "      valid   = 7,838"
)

print(
    "      missing = 1,924"
)

print(
    "      zeros   = 1"
)


# ============================================================
# 14. DIETARY WEIGHT CHECK
# ============================================================

#
# WTDRD1:
# valid = 9,255
#
# WTDR2D:
# valid = 7,838
#
# KEEP BOTH AS SOURCE FLOATS.
#

if int(
    df["WTDRD1"]
    .notna()
    .sum()
) != 9255:

    raise ValueError(
        "WTDRD1 valid count differs."
    )


if int(
    df["WTDR2D"]
    .notna()
    .sum()
) != 7838:

    raise ValueError(
        "WTDR2D valid count differs."
    )


if not (
    df["WTDRD1"]
    .dropna()
    .gt(0)
    .all()
):

    raise ValueError(
        "Unexpected WTDRD1 value."
    )


if not (
    df["WTDR2D"]
    .dropna()
    .gt(0)
    .all()
):

    raise ValueError(
        "Unexpected WTDR2D value."
    )


print(
    "\nPASS: Dietary weights validated."
)

print(
    "      WTDRD1 valid = 9,255"
)

print(
    "      WTDR2D valid = 7,838"
)


# ============================================================
# 15. DR2DRSTZ FREQUENCY CHECK
# ============================================================

EXPECTED_DR2DRSTZ = {
    1: 7715,
    2: 82,
    4: 123,
    5: 1842
}


actual_status = (
    df["DR2DRSTZ"]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .to_dict()
)


actual_status = {
    int(k): int(v)
    for k, v
    in actual_status.items()
}


if actual_status != EXPECTED_DR2DRSTZ:

    raise ValueError(
        "DR2DRSTZ frequency mismatch.\n"
        f"Expected: {EXPECTED_DR2DRSTZ}\n"
        f"Actual:   {actual_status}"
    )


print(
    "\nPASS: DR2DRSTZ matches CDC."
)

print(
    "      1 Reliable       = 7,715"
)

print(
    "      2 Not reliable   = 82"
)

print(
    "      4 Breast milk    = 123"
)

print(
    "      5 Not done       = 1,842"
)


# ============================================================
# 16. NUTRIENT VALID COUNT CHECK
# ============================================================

#
# All 65 nutrient variables have:
#
# Valid   = 7,715
# Missing = 2,047
#

nutrient_valid_failures = {}


for col in NUTRIENT_COLUMNS:

    valid_count = int(
        df[col]
        .notna()
        .sum()
    )


    if valid_count != 7715:

        nutrient_valid_failures[
            col
        ] = valid_count


if nutrient_valid_failures:

    raise ValueError(
        "Unexpected nutrient valid counts:\n"
        + str(
            nutrient_valid_failures
        )
    )


print(
    "\nPASS: All 65 nutrient variables "
    "have exactly 7,715 valid values."
)


# ============================================================
# 17. WATER VARIABLE CHECKS
# ============================================================

EXPECTED_WATER = {

    "DR2_320Z": {
        "valid": 7838,
        "missing": 1924,
        "min": 0,
        "max": 11376,
        "zeros": 2004
    },

    "DR2_330Z": {
        "valid": 7838,
        "missing": 1924,
        "min": 0,
        "max": 7584,
        "zeros": 4159
    },

    "DR2BWATZ": {
        "valid": 7838,
        "missing": 1924,
        "min": 0,
        "max": 11376,
        "zeros": 5202
    }
}


print(
    "\n--- WATER VARIABLE CHECK ---"
)


for col, spec in EXPECTED_WATER.items():

    s = df[col]


    if int(
        s.notna()
        .sum()
    ) != spec["valid"]:

        raise ValueError(
            f"{col}: valid count differs."
        )


    if int(
        s.isna()
        .sum()
    ) != spec["missing"]:

        raise ValueError(
            f"{col}: missing count differs."
        )


    if s.min() != spec["min"]:

        raise ValueError(
            f"{col}: minimum differs."
        )


    if s.max() != spec["max"]:

        raise ValueError(
            f"{col}: maximum differs."
        )


    if int(
        s.eq(0)
        .sum()
    ) != spec["zeros"]:

        raise ValueError(
            f"{col}: zero count differs."
        )


    print(
        f"PASS: {col:<9} "
        f"zero={spec['zeros']:,}"
    )


# ============================================================
# 18. DR2_300 FREQUENCY CHECK
# ============================================================

EXPECTED_DR2_300 = {
    1: 480,
    2: 6399,
    3: 981,
    9: 60
}


actual = (
    df["DR2_300"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_DR2_300:

    raise ValueError(
        "DR2_300 frequency mismatch."
    )


if int(
    df["DR2_300"]
    .isna()
    .sum()
) != 1842:

    raise ValueError(
        "DR2_300 missing count differs."
    )


print(
    "\nPASS: DR2_300 matches CDC."
)


# ============================================================
# 19. DR2TWS FREQUENCY CHECK
# ============================================================

EXPECTED_DR2TWS = {
    1: 5107,
    2: 649,
    3: 85,
    4: 1788,
    91: 6,
    99: 285
}


actual = (
    df["DR2TWS"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_DR2TWS:

    raise ValueError(
        "DR2TWS frequency mismatch."
    )


if int(
    df["DR2TWS"]
    .isna()
    .sum()
) != 1842:

    raise ValueError(
        "DR2TWS missing count differs."
    )


print(
    "PASS: DR2TWS matches CDC."
)


# ============================================================
# 20. FRACTIONAL VALUE CHECK
# ============================================================

#
# Every genuine fractional value must occur ONLY
# in an approved dietary measurement field.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        fractional_columns[
            col
        ] = count


unexpected_fractional_columns = (
    set(
        fractional_columns
    )
    -
    set(
        MEASUREMENT_COLUMNS
    )
)


if unexpected_fractional_columns:

    raise ValueError(
        "STOP: Genuine fractional values found "
        "outside dietary measurement fields:\n"
        + str(
            sorted(
                unexpected_fractional_columns
            )
        )
    )


if len(
    fractional_columns
) != 50:

    raise ValueError(
        "Unexpected number of variables "
        "containing genuine fractions."
    )


print(
    "\nPASS: Genuine fractional variables =",
    len(fractional_columns)
)

print(
    "PASS: All fractional values occur only "
    "inside approved dietary measurement fields."
)


# ============================================================
# 21. VERIFY INTEGER/CODE FIELDS ARE WHOLE NUMBERS
# ============================================================

integer_fraction_failures = {}


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        integer_fraction_failures[
            col
        ] = fractional_count


if integer_fraction_failures:

    raise ValueError(
        "STOP: Integer-designated fields contain "
        "fractional values:\n"
        + str(
            integer_fraction_failures
        )
    )


print(
    "PASS: All 13 identifier/code/count fields "
    "are whole-number values."
)


# ============================================================
# 22. BUILD EXPECTED SOURCE
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY the 57,565 validated tiny -> zero
# restorations.
#
# NOTHING ELSE IS MODIFIED.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 23. CLEAN INTEGER / CODE / COUNT FIELDS ONLY
# ============================================================

#
# Examples:
#
# SEQN
# 41475.0 -> 41475
#
# DR2DRSTZ
# 1.0 -> 1
#
# DR2DBIH
# 0.0 -> 0
#
# DR2TNUMF
# 0.0 -> 0
#
#
# DO NOT Int64 any dietary measurement field.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN/code/count fields "
    "converted to clean integer formatting."
)

print(
    "PASS: All 70 dietary measurement fields "
    "left as source-decoded floats."
)


# ============================================================
# 24. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in MEASUREMENT_COLUMNS:

        current_values = (
            df[col]
            .to_numpy()
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "\nPASS: Pre-export numeric fidelity exact."
)


# ============================================================
# 25. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO round()
# NO float_format
# NO decimal quantization
# NO recalculation
#
# Let pandas write source-decoded float values.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 26. RE-READ WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 27. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 28. EXACT NUMERIC XPT -> CSV FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 83 CSV columns numerically "
    "match expected XPT values exactly."
)


# ============================================================
# 29. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 30. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 9762:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 31. STRICT RAW CSV FIDELITY
# ============================================================

#
# Integer/code/count fields:
#
# str(int(source_value))
#
#
# Dietary measurements:
#
# str(float(source_value))
#
# This preserves long dietary decimal representations.
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in MEASUREMENT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "DR2TOT_E raw CSV fidelity failure."
    )


print(
    "PASS: Every raw CSV cell matches "
    "the intended XPT-derived representation."
)


# ============================================================
# 32. UNWANTED .0 CHECK
#    INTEGER FIELDS ONLY
# ============================================================

#
# Do NOT globally reject ".0".
#
# Dietary values such as:
#
# 1800.0
# 0.0
# 1.0
#
# are allowed and should remain.
#

unwanted_dot_zero_count = 0
unwanted_dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = (
            row[col]
        )


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            unwanted_dot_zero_count += 1


            if len(
                unwanted_dot_zero_examples
            ) < 20:

                unwanted_dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nUnwanted integer '.0' values:",
    unwanted_dot_zero_count
)


if unwanted_dot_zero_count != 0:

    print(
        unwanted_dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 remains in "
        "SEQN/code/count fields."
    )


print(
    "PASS: No unwanted .0 in integer fields."
)


# ============================================================
# 33. STRICT DIETARY DECIMAL TOKEN CHECK
# ============================================================

#
# Every measurement value is compared against:
#
# str(float(original_XPT_value))
#
# Examples:
#
# 72.54 remains 72.54
#
# 1.0 remains 1.0
#
# 0.028999999999999998 remains
# 0.028999999999999998
#

measurement_raw_mismatches = 0
measurement_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                float(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            measurement_raw_mismatches += 1


            if len(
                measurement_examples
            ) < 20:

                measurement_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "Dietary measurement raw differences:",
    measurement_raw_mismatches
)


if measurement_raw_mismatches != 0:

    print(
        measurement_examples
    )

    raise ValueError(
        "Dietary decimal representation changed."
    )


print(
    "PASS: Every dietary measurement "
    "is preserved AS IS."
)


# ============================================================
# 34. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 35. FINAL TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains in CSV."
)


# ============================================================
# 36. FINAL ZERO COUNT CHECK
# ============================================================

zero_failures = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:

        zero_failures[col] = {
            "expected": expected_count,
            "actual": actual_count
        }


if zero_failures:

    raise ValueError(
        "Final zero counts changed:\n"
        + str(
            zero_failures
        )
    )


print(
    "PASS: All 57,565 restored zero values "
    "remain correct after CSV export."
)


# ============================================================
# 37. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("DR2TOT_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nOriginal tiny artifacts:",
    f"{EXPECTED_TINY_TOTAL:,}"
)

print(
    "Variables affected:",
    len(
        EXPECTED_TINY_COUNTS
    )
)

print(
    "Remaining tiny artifacts:",
    remaining_tiny
)


print(
    "\nSpecial integer zeros:"
)

print(
    "DR2DBIH = 5"
)

print(
    "DR2TNUMF = 1"
)


print(
    "\nDietary measurement fields preserved AS IS:",
    len(
        MEASUREMENT_COLUMNS
    )
)

print(
    "Nutrient fields preserved AS IS:",
    len(
        NUTRIENT_COLUMNS
    )
)

print(
    "Variables with genuine fractional values:",
    len(
        fractional_columns
    )
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Dietary measurement raw differences:",
    measurement_raw_mismatches
)

print(
    "Unwanted integer .0 values:",
    unwanted_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "DR2TOT_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 DR2TOT_E XPT -> CSV
Rows: 9,762
Columns: 83
PASS: DR2TOT_E structure = 9,762 rows x 83 columns.

Nutrient fields preserved AS IS: 65
Total dietary measurement fields: 70
Identifier/code/count fields: 13

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 57,565

Tiny values by variable:
DR2TATOA    6871
DR2TALCO    6869
DR2BWATZ    5202
DR2TP184    4939
DR2TB12A    4893
DR2_330Z    4159
DR2TTHEO    4152
DR2TP225    2898
DR2TM221    2502
DR2TLYCO    2275
DR2TP205    2174
DR2TCAFF    2118
DR2_320Z    2004
DR2TP226    1744
DR2TACAR     989
DR2TCRYP     835
DR2TS040     509
DR2TS060     486
DR2TP204     301
DR2TS080     269
DR2TFA       153
DR2TS100     123
DR2TBCAR     119
DR2TLZ       110
DR2TVD       100
DR2TM201      92
DR2TFIBE      83
DR2TRET       77
DR2TFF        76
DR2TCHOL      55
DR2TVC        43
DR2TS120      33
DR2TM161      30
DR2TVB12      25
DR2TVARA      24
DR2TVK        12
DR2TS140      11
DR2TS180      10
DR2TM181       9
DR